# Chapter 9 - Lab 1: <font color='blue'>Sequential Claims Pipeline with Compliance Guardrail</font>

**<font color='purple'>Goal</font>**:
Build a **7-agent sequential pipeline** that processes a real auto-collision claim from First Notice of Loss to final compliance review. The pipeline is the **Sequential Pipeline with Guardrails** architecture from Chapter 7 (§2.1) applied to insurance.

Seven agents pass control via explicit handoffs, share data through a typed `Context` object, and write to an audit trail at every step. The seventh agent — Compliance — has *veto power* and can block a decision that fails regulatory checks.

By the end you will see an entire claims workflow run end-to-end on a sample claim, with every agent's contribution recorded for audit.

**<font color='purple'>Tech stack</font>**:

* **LlamaIndex** — `FunctionAgent`, `AgentWorkflow`, `Context`.
* **OpenAI** `gpt-4o`.
* **Pydantic** — typed contracts for the claim record and helpers from `common.py`.

## 1. Install packages

In [ ]:
%pip install -q llama-index llama-index-llms-openai pydantic python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally — assume the env var is already set.
    pass

## 3. Bootstrap the shared models and helpers

Chapter 9 labs share a `common.py` module that defines the Pydantic models and helpers used across the chapter. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [ ]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%209/common.py',
        'common.py',
    )

from common import (
    ClaimType, ClaimStatus,
    SAMPLE_CLAIM, SAMPLE_POLICY, SAMPLE_DOCUMENTS,
    generate_claim_id, log_audit, days_between,
)

print('Common helpers loaded.')

## 4. Define the seven pipeline tools

Each agent has a dedicated async tool that reads from and writes to the shared `Context`. This is the same shared-state pattern from Chapter 7 — but here every write also calls `log_audit` to record what happened, when, and which agent did it. The audit log is the most important artifact for compliance: any decision must be reconstructable from it.

In [ ]:
from llama_index.core.workflow import Context

def classify_claim_type(description: str) -> ClaimType:
    d = description.lower()
    if any(w in d for w in ['vehicle', 'car', 'truck', 'collision', 'bumper']):
        return ClaimType.AUTO
    if any(w in d for w in ['house', 'roof', 'flood', 'fire', 'property']):
        return ClaimType.PROPERTY
    if any(w in d for w in ['hospital', 'medical', 'surgery', 'injury']):
        return ClaimType.HEALTH
    return ClaimType.LIABILITY


async def process_fnol(ctx: Context, claim_data: dict) -> str:
    claim_id = generate_claim_id()
    ct = classify_claim_type(claim_data.get('description', ''))
    claim = {
        'claim_id': claim_id,
        'policyholder_name': claim_data.get('name', ''),
        'policy_number': claim_data.get('policy_number', ''),
        'claim_type': ct.value,
        'date_of_loss': claim_data.get('date_of_loss', ''),
        'description': claim_data.get('description', ''),
        'status': 'received', 'audit_log': [],
    }
    state = await ctx.get('state')
    state['claim'] = claim
    log_audit(state, 'FNOLAgent', 'Claim created', f'ID: {claim_id}, Type: {ct.value}')
    await ctx.set('state', state)
    return f'FNOL processed. Claim {claim_id}.'


async def parse_documents(ctx: Context) -> str:
    state = await ctx.get('state')
    claim = state['claim']
    docs = state.get('submitted_documents', [])
    parsed, damage_parts = [], []
    for d in docs:
        if d.get('type') == 'photo':
            desc = d.get('description', '')
            parsed.append({'type': 'damage_photo', 'damage_description': desc})
            damage_parts.append(desc)
        elif d.get('type') == 'claim_form':
            parsed.append({'type': 'claim_form', 'fields': d.get('fields', {})})
        elif d.get('type') == 'police_report':
            parsed.append({'type': 'police_report', 'summary': d.get('summary', '')})
    claim['parsed_documents'] = parsed
    claim['damage_description'] = ' '.join(damage_parts)
    claim['status'] = 'parsing'
    log_audit(state, 'DocumentParser', f'Parsed {len(parsed)} docs')
    await ctx.set('state', state)
    return f'Parsed {len(parsed)} documents.'

**Coverage Agent.** Checks whether the loss falls within the policy period and is not excluded. Writes back the coverage flag, limit, deductible, and reasoning.

In [ ]:
async def verify_coverage(ctx: Context) -> str:
    state = await ctx.get('state')
    claim, policy = state['claim'], state.get('policy', {})
    is_covered, exclusions, reasoning = True, [], []
    dol = claim.get('date_of_loss', '')
    inc, exp = policy.get('inception_date', ''), policy.get('expiration_date', '')
    if dol and inc and exp and (dol < inc or dol > exp):
        is_covered = False
        reasoning.append(f'Loss date {dol} outside policy period {inc}-{exp}.')
    for ex in policy.get('exclusions', []):
        if ex.lower() in claim.get('description', '').lower():
            exclusions.append(ex)
    if is_covered and not exclusions:
        reasoning.append(
            f"Event covered under {policy.get('coverage_sections','comprehensive')}. "
            f"Limit: ${policy.get('coverage_limit',0):,.0f}. "
            f"Deductible: ${policy.get('deductible',0):,.0f}."
        )
    claim['coverage_verified'] = is_covered and not exclusions
    claim['applicable_limit'] = policy.get('coverage_limit', 0)
    claim['deductible']       = policy.get('deductible', 0)
    claim['exclusions_found'] = exclusions
    claim['coverage_reasoning'] = ' '.join(reasoning)
    claim['status'] = 'coverage_check'
    log_audit(state, 'CoverageAgent', 'verified' if claim['coverage_verified'] else 'denied')
    await ctx.set('state', state)
    return 'Coverage check done.'

**Fraud Screener.** Adds points for four classic red flags: short time since policy inception, multiple prior claims, claim near the coverage limit, and late reporting. A score above 70 escalates to the deeper fraud investigation pattern (see Lab 2).

In [ ]:
async def screen_for_fraud(ctx: Context) -> str:
    state = await ctx.get('state')
    claim, policy = state['claim'], state.get('policy', {})
    flags, score = [], 0
    dol, inc = claim.get('date_of_loss', ''), policy.get('inception_date', '')
    if dol and inc and days_between(inc, dol) < 30:
        flags.append(f'Claim filed {days_between(inc, dol)} days after inception')
        score += 25
    if len(state.get('claims_history', [])) >= 3:
        flags.append(f"{len(state['claims_history'])} prior claims")
        score += 20
    if dol and days_between(dol, state.get('fnol_date', dol)) > 30:
        flags.append('Late reporting (>30 days)')
        score += 15
    score = min(score, 100)
    claim['fraud_score'] = score
    claim['red_flags'] = flags
    claim['fraud_recommendation'] = 'investigate' if score > 70 else 'proceed'
    claim['status'] = 'fraud_screening'
    log_audit(state, 'FraudScreener', f'Score {score}/100')
    await ctx.set('state', state)
    return f'Fraud score: {score}/100.'

**Assessment Agent.** Estimates damages from the parsed description (a vision model would do better in production), applies the deductible, and caps at the coverage limit.

In [ ]:
async def assess_damage(ctx: Context) -> str:
    state = await ctx.get('state')
    claim = state['claim']
    desc = claim.get('damage_description', '').lower()
    deductible, limit = claim.get('deductible', 500), claim.get('applicable_limit', 50_000)
    base = 2_000
    if 'bumper' in desc: base = 3_500
    if 'tail light' in desc: base += 800
    if 'airbag' in desc: base += 5_000
    if 'total' in desc: base = limit * 0.8
    payout = max(0, min(base - deductible, limit))
    claim['estimated_payout'] = payout
    claim['assessment_breakdown'] = (
        f'Repair: ${base:,.0f}. Deductible: ${deductible:,.0f}. '
        f'Limit: ${limit:,.0f}. Net: ${payout:,.0f}.'
    )
    claim['status'] = 'assessment'
    log_audit(state, 'AssessmentAgent', f'Payout ${payout:,.0f}')
    await ctx.set('state', state)
    return f'Estimated payout ${payout:,.0f}.'

**Decision Agent.** Three-way decision: DENIED if coverage failed, ESCALATED if fraud score crossed the threshold, otherwise APPROVED. Reasoning is verbose by design — it must reconstruct the entire chain for audit.

In [ ]:
async def make_decision(ctx: Context) -> str:
    state = await ctx.get('state')
    claim = state['claim']
    if not claim.get('coverage_verified'):
        decision = 'DENIED'
        reasoning = f"Coverage not verified: {claim.get('coverage_reasoning','')}. Exclusions: {claim.get('exclusions_found',[])}."
    elif claim.get('fraud_recommendation') == 'investigate':
        decision = 'ESCALATED'
        reasoning = f'Escalated for fraud investigation. Score: {claim.get("fraud_score",0)}. Flags: {claim.get("red_flags",[])}.'
    else:
        decision = 'APPROVED'
        reasoning = (
            f'Approved. Coverage: {claim.get("coverage_reasoning","")}. '
            f'Fraud score: {claim.get("fraud_score",0)} (low risk). '
            f'Payout: ${claim.get("estimated_payout",0):,.0f}.'
        )
    claim['decision'] = decision
    claim['decision_reasoning'] = reasoning
    claim['status'] = decision.lower()
    log_audit(state, 'DecisionAgent', decision)
    await ctx.set('state', state)
    return f'Decision: {decision}.'

**Compliance Agent (guardrail).** Validates the decision against three regulatory checks: the denial reason must be valid; the reasoning must be long enough to audit; and every required pipeline field must be populated. If any check fails, the decision is BLOCKED.

In [ ]:
async def validate_compliance(ctx: Context) -> str:
    state = await ctx.get('state')
    claim = state['claim']
    issues = []
    if claim.get('decision') == 'DENIED':
        valid_reasons = ['not covered', 'exclusion', 'policy lapsed', 'fraud', 'outside policy period']
        if not any(r in claim.get('decision_reasoning', '').lower() for r in valid_reasons):
            issues.append('Denial reason invalid under regulation.')
    if len(claim.get('decision_reasoning', '')) < 50:
        issues.append('Reasoning too short for audit.')
    for field in ('coverage_verified', 'fraud_score', 'estimated_payout'):
        if claim.get(field) is None:
            issues.append(f'Missing required field: {field}')
    if issues:
        claim['status'] = 'compliance_review'
        log_audit(state, 'ComplianceAgent', 'BLOCKED', str(issues))
        await ctx.set('state', state)
        return f'BLOCKED: {issues}'
    log_audit(state, 'ComplianceAgent', 'Compliant')
    await ctx.set('state', state)
    return 'Compliance check passed.'

## 5. Define the seven pipeline agents

Each agent's `system_prompt` is a tight job description; `can_handoff_to` names the **single next agent** in the pipeline. The Compliance agent can hand off back to FNOL — it can restart the whole pipeline if it finds a fundamental defect.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

llm = OpenAI(model='gpt-4o', temperature=0)

fnol_agent = FunctionAgent(
    name='FNOLAgent', description='Process first notice of loss.',
    system_prompt='You handle initial claim intake. Classify and create the record, then hand off to DocumentParser.',
    llm=llm, tools=[process_fnol], can_handoff_to=['DocumentParser'],
)
parser_agent = FunctionAgent(
    name='DocumentParser', description='Parse submitted documents.',
    system_prompt='Parse claim forms, photos, and police reports. Hand off to CoverageAgent.',
    llm=llm, tools=[parse_documents], can_handoff_to=['CoverageAgent'],
)
coverage_agent = FunctionAgent(
    name='CoverageAgent', description='Verify policy coverage.',
    system_prompt='Check limits, deductibles, and exclusions. Hand off to FraudScreener.',
    llm=llm, tools=[verify_coverage], can_handoff_to=['FraudScreener'],
)
fraud_agent = FunctionAgent(
    name='FraudScreener', description='Score for fraud indicators.',
    system_prompt='Score the claim against red flags and recommend proceed or investigate. Hand off to AssessmentAgent.',
    llm=llm, tools=[screen_for_fraud], can_handoff_to=['AssessmentAgent'],
)
assessment_agent = FunctionAgent(
    name='AssessmentAgent', description='Estimate the payout.',
    system_prompt='Estimate damages, apply the deductible, cap at the coverage limit. Hand off to DecisionAgent.',
    llm=llm, tools=[assess_damage], can_handoff_to=['DecisionAgent'],
)
decision_agent = FunctionAgent(
    name='DecisionAgent', description='Make the final decision.',
    system_prompt='Approve, deny, or escalate. Provide a full reasoning chain. Hand off to ComplianceAgent.',
    llm=llm, tools=[make_decision], can_handoff_to=['ComplianceAgent'],
)
compliance_agent = FunctionAgent(
    name='ComplianceAgent', description='Regulatory guardrail with veto power.',
    system_prompt=(
        'Validate the decision against regulation. You have VETO power — block any '
        'non-compliant decision. If a fundamental defect is found, restart from FNOLAgent.'
    ),
    llm=llm, tools=[validate_compliance], can_handoff_to=['FNOLAgent'],
)

## 6. Assemble and run the workflow

Initialise the workflow's shared state with the sample policy, documents, and FNOL date from `common.py`. Then send the sample claim through and stream events to watch each agent take its turn.

In [ ]:
import json
from llama_index.core.agent.workflow import AgentWorkflow

workflow = AgentWorkflow(
    agents=[fnol_agent, parser_agent, coverage_agent, fraud_agent,
            assessment_agent, decision_agent, compliance_agent],
    root_agent='FNOLAgent',
    initial_state={
        'claim': {}, 'policy': SAMPLE_POLICY,
        'submitted_documents': SAMPLE_DOCUMENTS,
        'claims_history': [],
        'fnol_date': '2026-04-08',
    },
)

handler = workflow.run(user_msg=json.dumps(SAMPLE_CLAIM))

async for event in handler.stream_events():
    if hasattr(event, 'current_agent_name'):
        print(f'\n[Active: {event.current_agent_name}]')

result = await handler
print('\nFinal result:\n' + str(result))

## 7. Inspect the audit trail

The audit log is the single most important artifact of this pipeline. If the claim is later disputed or audited, this log must reconstruct every decision the system made.

We pull the final claim record out of the workflow's context. In LlamaIndex AgentWorkflow the context is available on the handler *after* the workflow has finished — we access it with `handler.ctx.get(...)`.

In [ ]:
final_state = await handler.ctx.get('state')
claim = final_state.get('claim', {}) if final_state else {}

print(f'Claim ID: {claim.get("claim_id")}')
print(f'Decision: {claim.get("decision")}')
print(f'Payout:   ${claim.get("estimated_payout", 0):,.0f}')
print(f'Status:   {claim.get("status")}\n')
print('=' * 50)
print('  AUDIT TRAIL')
print('=' * 50)
for entry in claim.get('audit_log', []):
    print(f'  {entry["timestamp"]} [{entry["agent"]}] {entry["action"]}')
    if entry.get('detail'):
        print(f'      {entry["detail"][:120]}')

## 8. Results

You processed a real-world auto-collision claim through seven specialist agents and produced a fully auditable decision. **What to notice about the claims-pipeline architecture:**

* **Compliance is an agent, not an afterthought.** A guardrail agent with veto power gives you a   single, testable place to enforce regulation.
* **Audit trails are written by tools, not by agents.** Every tool calls `log_audit` as it   mutates state — the trail is a byproduct, not extra work.
* **Pydantic models pay off.** `ClaimRecord`, `PolicyRecord`, and friends keep the contract   between agents explicit; a mismatch surfaces at the next agent's tool call.
* **The same architecture extends to property, health, or liability claims** by swapping the tools,   not the agent topology.